In [1]:
from pyspark.sql import functions as F


# ============================================================
# AirOps 360 - Week 4 Task 22
# Weather-to-flight enrichment + cardinality QA
#
# INPUT FLIGHT GRAIN:
#   1 row = 1 accepted scheduled flight occurrence
#
# INPUT WEATHER GRAIN:
#   1 row = 1 airport + 1 local observation hour
#
# OUTPUT GRAIN:
#   MUST remain 1 row = 1 flight_key
#
# CURRENT WEATHER SCOPE:
#   ORD + ATL
#   April 2026 only
# ============================================================


# ------------------------------------------------------------
# 0. CONFIGURATION
# ------------------------------------------------------------

spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)

FLIGHT_TABLE = "slv_flights_validated"
WEATHER_TABLE = "slv_weather_hourly"

TARGET_TABLE = "slv_flights_weather_enriched"
METRICS_TABLE = "slv_flights_weather_enrichment_metrics"

EXPECTED_FLIGHT_ROWS = 597_919
EXPECTED_WEATHER_ROWS = 1_440

WEATHER_PILOT_AIRPORTS = ["ORD", "ATL"]

ENRICHMENT_VERSION = "weather_enrichment_v1"


print("TASK 22 CONFIGURATION")
print("---------------------")
print("Flight source :", FLIGHT_TABLE)
print("Weather source:", WEATHER_TABLE)
print("Target        :", TARGET_TABLE)
print("Metrics       :", METRICS_TABLE)
print("Weather pilot :", WEATHER_PILOT_AIRPORTS)


# ============================================================
# 1. READ ACCEPTED FLIGHTS + VALIDATED WEATHER
# ============================================================

flights = spark.table(
    FLIGHT_TABLE
)

weather = spark.table(
    WEATHER_TABLE
)


flight_rows_before = flights.count()

flight_keys_before = (
    flights
    .select("flight_key")
    .distinct()
    .count()
)

weather_rows = weather.count()


print()
print(
    f"Accepted flight rows:       "
    f"{flight_rows_before:,}"
)

print(
    f"Distinct flight_key values: "
    f"{flight_keys_before:,}"
)

print(
    f"Weather rows:               "
    f"{weather_rows:,}"
)


assert (
    flight_rows_before
    ==
    EXPECTED_FLIGHT_ROWS
), (
    f"STOP: expected "
    f"{EXPECTED_FLIGHT_ROWS:,} flights, "
    f"found {flight_rows_before:,}"
)


assert (
    flight_keys_before
    ==
    flight_rows_before
), (
    "STOP: accepted flight grain "
    "is not unique before enrichment"
)


assert (
    weather_rows
    ==
    EXPECTED_WEATHER_ROWS
), (
    f"STOP: expected "
    f"{EXPECTED_WEATHER_ROWS:,} weather rows, "
    f"found {weather_rows:,}"
)


print(
    "\nSOURCE-GRAIN VALIDATION: PASS"
)


# ============================================================
# 2. PROVE WEATHER SIDE IS UNIQUE BEFORE JOIN
#
# This is the most important pre-join cardinality gate.
# ============================================================

weather_duplicate_groups = (
    weather
    .groupBy(
        "airport_code",
        "weather_hour_local",
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)


weather_distinct_keys = (
    weather
    .select(
        "weather_key"
    )
    .distinct()
    .count()
)


weather_airports = {
    r["airport_code"]
    for r in (
        weather
        .select("airport_code")
        .distinct()
        .collect()
    )
}


print()
print(
    "Weather duplicate airport-hour "
    f"groups: {weather_duplicate_groups}"
)

print(
    "Weather distinct keys: "
    f"{weather_distinct_keys:,}"
)

print(
    "Weather airports:",
    weather_airports,
)


assert (
    weather_duplicate_groups == 0
), (
    "STOP: weather contains duplicate "
    "airport-hour rows"
)


assert (
    weather_distinct_keys
    ==
    weather_rows
), (
    "STOP: weather_key is not unique"
)


assert (
    weather_airports
    ==
    set(WEATHER_PILOT_AIRPORTS)
), (
    "STOP: unexpected weather airport scope: "
    f"{weather_airports}"
)


print(
    "WEATHER UNIQUENESS GATE: PASS"
)


# ============================================================
# 3. BUILD THE FLIGHT'S SCHEDULED DEPARTURE LOCAL TIMESTAMP
#
# BTS CRSDepTime is HHMM.
#
# Examples:
#   818  -> 08:18
#   45   -> 00:45
#   1630 -> 16:30
#
# Special BTS case:
#   2400 -> 00:00 on NEXT calendar day
# ============================================================

crs_time = F.col(
    "crs_dep_time_hhmm"
)


flights_with_join_time = (
    flights

    # ---------------------------------------------
    # Normalize 2400 to 0000
    # ---------------------------------------------

    .withColumn(
        "_dep_hhmm_normalized",

        F.when(
            crs_time == 2400,
            F.lit(0)
        )
        .otherwise(
            crs_time
        )
        .cast("int")
    )

    # ---------------------------------------------
    # 2400 belongs to next calendar day
    # ---------------------------------------------

    .withColumn(
        "_dep_service_date",

        F.when(
            crs_time == 2400,
            F.date_add(
                F.col("flight_date"),
                1
            )
        )
        .otherwise(
            F.col("flight_date")
        )
    )

    # ---------------------------------------------
    # Extract hour
    # ---------------------------------------------

    .withColumn(
        "_dep_hour",

        F.floor(
            F.col("_dep_hhmm_normalized")
            /
            F.lit(100)
        ).cast("int")
    )

    # ---------------------------------------------
    # Extract minute
    # ---------------------------------------------

    .withColumn(
        "_dep_minute",

        (
            F.col("_dep_hhmm_normalized")
            %
            F.lit(100)
        ).cast("int")
    )

    # ---------------------------------------------
    # Construct scheduled local timestamp
    # ---------------------------------------------

    .withColumn(
        "origin_sched_dep_local",

        F.to_timestamp(

            F.concat(

                F.date_format(
                    F.col("_dep_service_date"),
                    "yyyy-MM-dd"
                ),

                F.lit(" "),

                F.lpad(
                    F.col("_dep_hour")
                    .cast("string"),
                    2,
                    "0"
                ),

                F.lit(":"),

                F.lpad(
                    F.col("_dep_minute")
                    .cast("string"),
                    2,
                    "0"
                ),

                F.lit(":00")
            ),

            "yyyy-MM-dd HH:mm:ss"
        )
    )

    # ---------------------------------------------
    # Weather is hourly, so truncate flight to hour
    # ---------------------------------------------

    .withColumn(
        "origin_sched_dep_hour_local",

        F.date_trunc(
            "hour",
            F.col(
                "origin_sched_dep_local"
            )
        )
    )
)


# ============================================================
# 4. VALIDATE FLIGHT JOIN-TIME DERIVATION
# ============================================================

derived_timestamp_nulls = (
    flights_with_join_time
    .filter(
        F.col(
            "origin_sched_dep_local"
        ).isNull()
        |
        F.col(
            "origin_sched_dep_hour_local"
        ).isNull()
    )
    .count()
)


bad_derived_minutes = (
    flights_with_join_time
    .filter(
        (F.col("_dep_hour") < 0)
        |
        (F.col("_dep_hour") > 23)
        |
        (F.col("_dep_minute") < 0)
        |
        (F.col("_dep_minute") > 59)
    )
    .count()
)


midnight_2400_rows = (
    flights
    .filter(
        F.col("crs_dep_time_hhmm") == 2400
    )
    .count()
)


print()
print(
    "Derived timestamp NULL rows:",
    derived_timestamp_nulls,
)

print(
    "Invalid derived HH:MM rows:",
    bad_derived_minutes,
)

print(
    "Source CRSDepTime=2400 rows:",
    midnight_2400_rows,
)


assert (
    derived_timestamp_nulls == 0
), (
    "STOP: scheduled local timestamps "
    "could not be derived"
)


assert (
    bad_derived_minutes == 0
), (
    "STOP: derived scheduled time "
    "contains invalid hour/minute"
)


print(
    "FLIGHT TIME DERIVATION: PASS"
)


# ============================================================
# 5. PREPARE WEATHER WITH PREFIXED COLUMN NAMES
#
# Avoid collisions between flight lineage and weather lineage.
# ============================================================

weather_for_join = (
    weather
    .select(

        F.col(
            "weather_key"
        ).alias(
            "origin_weather_key"
        ),

        F.col(
            "airport_code"
        ).alias(
            "wx_airport_code"
        ),

        F.col(
            "weather_hour_local"
        ).alias(
            "origin_weather_hour_local"
        ),

        F.col(
            "weather_hour_utc"
        ).alias(
            "origin_weather_hour_utc"
        ),

        F.col(
            "response_timezone"
        ).alias(
            "origin_weather_timezone"
        ),

        F.col(
            "temperature_2m_c"
        ).alias(
            "origin_temperature_2m_c"
        ),

        F.col(
            "relative_humidity_2m_pct"
        ).alias(
            "origin_relative_humidity_2m_pct"
        ),

        F.col(
            "precipitation_mm"
        ).alias(
            "origin_precipitation_mm"
        ),

        F.col(
            "snowfall_cm"
        ).alias(
            "origin_snowfall_cm"
        ),

        F.col(
            "weather_code"
        ).alias(
            "origin_weather_code"
        ),

        F.col(
            "cloud_cover_pct"
        ).alias(
            "origin_cloud_cover_pct"
        ),

        F.col(
            "wind_speed_10m_kmh"
        ).alias(
            "origin_wind_speed_10m_kmh"
        ),

        F.col(
            "wind_direction_10m_deg"
        ).alias(
            "origin_wind_direction_10m_deg"
        ),

        F.col(
            "_bronze_batch_key"
        ).alias(
            "origin_weather_bronze_batch_key"
        ),

        F.col(
            "_bronze_run_id"
        ).alias(
            "origin_weather_bronze_run_id"
        ),

        F.col(
            "_bronze_load_id"
        ).alias(
            "origin_weather_bronze_load_id"
        ),

        F.col(
            "_silver_version"
        ).alias(
            "origin_weather_silver_version"
        ),
    )
)


# ============================================================
# 6. CARDINALITY-SAFE LEFT JOIN
#
# JOIN KEY:
#
# flight.origin
#     =
# weather.airport_code
#
# AND
#
# flight scheduled departure local hour
#     =
# weather local observation hour
# ============================================================

f = flights_with_join_time.alias("f")
w = weather_for_join.alias("w")


joined = (
    f
    .join(
        w,

        (
            F.upper(
                F.trim(
                    F.col("f.origin")
                )
            )
            ==
            F.col("w.wx_airport_code")
        )
        &
        (
            F.col(
                "f.origin_sched_dep_hour_local"
            )
            ==
            F.col(
                "w.origin_weather_hour_local"
            )
        ),

        "left"
    )

    .select(

        "f.*",

        F.col(
            "w.origin_weather_key"
        ),

        F.col(
            "w.wx_airport_code"
        ),

        F.col(
            "w.origin_weather_hour_local"
        ),

        F.col(
            "w.origin_weather_hour_utc"
        ),

        F.col(
            "w.origin_weather_timezone"
        ),

        F.col(
            "w.origin_temperature_2m_c"
        ),

        F.col(
            "w.origin_relative_humidity_2m_pct"
        ),

        F.col(
            "w.origin_precipitation_mm"
        ),

        F.col(
            "w.origin_snowfall_cm"
        ),

        F.col(
            "w.origin_weather_code"
        ),

        F.col(
            "w.origin_cloud_cover_pct"
        ),

        F.col(
            "w.origin_wind_speed_10m_kmh"
        ),

        F.col(
            "w.origin_wind_direction_10m_deg"
        ),

        F.col(
            "w.origin_weather_bronze_batch_key"
        ),

        F.col(
            "w.origin_weather_bronze_run_id"
        ),

        F.col(
            "w.origin_weather_bronze_load_id"
        ),

        F.col(
            "w.origin_weather_silver_version"
        ),
    )
)


# ============================================================
# 7. EXPLICITLY CLASSIFY WEATHER MATCH STATUS
#
# Current pilot weather only covers ORD and ATL.
#
# Therefore:
#
# MATCHED
#   exact airport/hour weather found
#
# OUTSIDE_WEATHER_PILOT_AIRPORT
#   origin is not ORD or ATL
#
# PILOT_AIRPORT_HOUR_UNMATCHED
#   origin is ORD/ATL but exact requested hour is
#   outside/missing from current weather pilot
# ============================================================

enriched = (
    joined

    .withColumn(
        "origin_weather_match_status",

        F.when(
            F.col(
                "origin_weather_key"
            ).isNotNull(),

            F.lit("MATCHED")
        )

        .when(
            ~F.upper(
                F.trim(
                    F.col("origin")
                )
            ).isin(
                WEATHER_PILOT_AIRPORTS
            ),

            F.lit(
                "OUTSIDE_WEATHER_PILOT_AIRPORT"
            )
        )

        .otherwise(
            F.lit(
                "PILOT_AIRPORT_HOUR_UNMATCHED"
            )
        )
    )

    .withColumn(
        "_weather_enrichment_version",
        F.lit(
            ENRICHMENT_VERSION
        )
    )

    .withColumn(
        "_weather_enriched_at_utc",
        F.current_timestamp()
    )
)


# ============================================================
# 8. POST-JOIN CARDINALITY QA
# ============================================================

flight_rows_after = enriched.count()


flight_keys_after = (
    enriched
    .select(
        "flight_key"
    )
    .distinct()
    .count()
)


duplicate_flight_groups_after = (
    enriched
    .groupBy(
        "flight_key"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)


print()
print(
    f"Flight rows before join: "
    f"{flight_rows_before:,}"
)

print(
    f"Flight rows after join:  "
    f"{flight_rows_after:,}"
)

print(
    f"Distinct flight_key before: "
    f"{flight_keys_before:,}"
)

print(
    f"Distinct flight_key after:  "
    f"{flight_keys_after:,}"
)

print(
    "Duplicate flight_key groups after join:",
    duplicate_flight_groups_after,
)


assert (
    flight_rows_after
    ==
    flight_rows_before
), (
    "STOP: weather join multiplied or dropped "
    "flight rows"
)


assert (
    flight_keys_after
    ==
    flight_keys_before
), (
    "STOP: distinct flight_key count changed"
)


assert (
    duplicate_flight_groups_after == 0
), (
    "STOP: one or more flights appear "
    "multiple times after weather join"
)


print(
    "POST-JOIN FLIGHT-GRAIN QA: PASS"
)


# ============================================================
# 9. VALIDATE MATCH CORRECTNESS
# ============================================================

wrong_airport_matches = (
    enriched
    .filter(
        F.col(
            "origin_weather_key"
        ).isNotNull()
        &
        (
            F.upper(
                F.trim(
                    F.col("origin")
                )
            )
            !=
            F.col("wx_airport_code")
        )
    )
    .count()
)


wrong_hour_matches = (
    enriched
    .filter(
        F.col(
            "origin_weather_key"
        ).isNotNull()
        &
        (
            F.col(
                "origin_sched_dep_hour_local"
            )
            !=
            F.col(
                "origin_weather_hour_local"
            )
        )
    )
    .count()
)


nonpilot_weather_matches = (
    enriched
    .filter(
        ~F.upper(
            F.trim(
                F.col("origin")
            )
        ).isin(
            WEATHER_PILOT_AIRPORTS
        )
        &
        F.col(
            "origin_weather_key"
        ).isNotNull()
    )
    .count()
)


assert (
    wrong_airport_matches == 0
), (
    f"STOP: wrong-airport matches = "
    f"{wrong_airport_matches}"
)


assert (
    wrong_hour_matches == 0
), (
    f"STOP: wrong-hour matches = "
    f"{wrong_hour_matches}"
)


assert (
    nonpilot_weather_matches == 0
), (
    f"STOP: non-pilot airports received "
    f"weather = {nonpilot_weather_matches}"
)


print(
    "WEATHER MATCH-CORRECTNESS QA: PASS"
)


# ============================================================
# 10. CALCULATE MATCH / UNMATCHED BEHAVIOR
# ============================================================

matched_rows = (
    enriched
    .filter(
        F.col(
            "origin_weather_key"
        ).isNotNull()
    )
    .count()
)


unmatched_rows = (
    flight_rows_after
    -
    matched_rows
)


pilot_origin_rows = (
    enriched
    .filter(
        F.upper(
            F.trim(
                F.col("origin")
            )
        ).isin(
            WEATHER_PILOT_AIRPORTS
        )
    )
    .count()
)


pilot_matched_rows = (
    enriched
    .filter(
        F.upper(
            F.trim(
                F.col("origin")
            )
        ).isin(
            WEATHER_PILOT_AIRPORTS
        )
        &
        F.col(
            "origin_weather_key"
        ).isNotNull()
    )
    .count()
)


pilot_unmatched_rows = (
    pilot_origin_rows
    -
    pilot_matched_rows
)


outside_pilot_rows = (
    flight_rows_after
    -
    pilot_origin_rows
)


print()
print(
    f"Weather matched flights:          "
    f"{matched_rows:,}"
)

print(
    f"Weather unmatched flights:        "
    f"{unmatched_rows:,}"
)

print(
    f"ORD/ATL origin flights:           "
    f"{pilot_origin_rows:,}"
)

print(
    f"ORD/ATL matched flights:          "
    f"{pilot_matched_rows:,}"
)

print(
    f"ORD/ATL unmatched flights:        "
    f"{pilot_unmatched_rows:,}"
)

print(
    f"Origins outside weather pilot:    "
    f"{outside_pilot_rows:,}"
)


assert (
    matched_rows
    +
    unmatched_rows
    ==
    flight_rows_after
)


assert (
    pilot_matched_rows
    +
    pilot_unmatched_rows
    ==
    pilot_origin_rows
)


print(
    "MATCH / UNMATCHED RECONCILIATION: PASS"
)


# ============================================================
# 11. SHOW UNMATCHED REASONS
# ============================================================

match_status_summary = (
    enriched
    .groupBy(
        "origin_weather_match_status"
    )
    .count()
    .orderBy(
        F.desc("count")
    )
)


print(
    "\nWeather match-status summary:"
)

display(
    match_status_summary
)


# ============================================================
# 12. WRITE CARDINALITY-SAFE ENRICHED SILVER
# ============================================================

(
    enriched.write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        TARGET_TABLE
    )
)


# ============================================================
# 13. WRITE ENRICHMENT QA METRICS
# ============================================================

metrics = (
    spark.range(1)

    .select(

        F.current_timestamp()
        .alias(
            "measured_at_utc"
        ),

        F.lit(
            ENRICHMENT_VERSION
        )
        .alias(
            "enrichment_version"
        ),

        F.lit(
            flight_rows_before
        )
        .cast("long")
        .alias(
            "flight_rows_before"
        ),

        F.lit(
            flight_rows_after
        )
        .cast("long")
        .alias(
            "flight_rows_after"
        ),

        F.lit(
            flight_keys_before
        )
        .cast("long")
        .alias(
            "distinct_flight_keys_before"
        ),

        F.lit(
            flight_keys_after
        )
        .cast("long")
        .alias(
            "distinct_flight_keys_after"
        ),

        F.lit(
            weather_rows
        )
        .cast("long")
        .alias(
            "weather_rows"
        ),

        F.lit(
            weather_duplicate_groups
        )
        .cast("long")
        .alias(
            "weather_duplicate_airport_hour_groups"
        ),

        F.lit(
            matched_rows
        )
        .cast("long")
        .alias(
            "weather_matched_flights"
        ),

        F.lit(
            unmatched_rows
        )
        .cast("long")
        .alias(
            "weather_unmatched_flights"
        ),

        F.lit(
            pilot_origin_rows
        )
        .cast("long")
        .alias(
            "pilot_origin_flights"
        ),

        F.lit(
            pilot_matched_rows
        )
        .cast("long")
        .alias(
            "pilot_origin_matched_flights"
        ),

        F.lit(
            pilot_unmatched_rows
        )
        .cast("long")
        .alias(
            "pilot_origin_unmatched_flights"
        ),

        F.lit(
            outside_pilot_rows
        )
        .cast("long")
        .alias(
            "outside_pilot_origin_flights"
        ),

        F.lit(
            wrong_airport_matches
        )
        .cast("long")
        .alias(
            "wrong_airport_matches"
        ),

        F.lit(
            wrong_hour_matches
        )
        .cast("long")
        .alias(
            "wrong_hour_matches"
        ),

        F.lit(
            duplicate_flight_groups_after
        )
        .cast("long")
        .alias(
            "duplicate_flight_groups_after"
        ),

        F.lit(
            flight_rows_before
            ==
            flight_rows_after
        )
        .alias(
            "row_count_preserved"
        ),

        F.lit(
            flight_keys_before
            ==
            flight_keys_after
        )
        .alias(
            "flight_key_grain_preserved"
        ),
    )
)


(
    metrics.write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        METRICS_TABLE
    )
)


# ============================================================
# 14. POST-WRITE REVALIDATION
# ============================================================

target = spark.table(
    TARGET_TABLE
)


target_rows = target.count()


target_distinct_keys = (
    target
    .select(
        "flight_key"
    )
    .distinct()
    .count()
)


target_duplicate_keys = (
    target
    .groupBy(
        "flight_key"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)


assert (
    target_rows
    ==
    flight_rows_before
)


assert (
    target_distinct_keys
    ==
    flight_keys_before
)


assert (
    target_duplicate_keys == 0
)


print(
    "\nPOST-WRITE TARGET QA: PASS"
)


# ============================================================
# 15. FINAL TASK 22 EVIDENCE
# ============================================================

print()
print("=" * 82)

print(
    "AIROPS 360 - TASK 22 "
    "WEATHER-TO-FLIGHT CARDINALITY-SAFE ENRICHMENT"
)

print("=" * 82)

print(
    f"Accepted flights before join:       "
    f"{flight_rows_before:,}"
)

print(
    f"Flights after weather join:         "
    f"{flight_rows_after:,}"
)

print(
    f"Distinct flight_key before:         "
    f"{flight_keys_before:,}"
)

print(
    f"Distinct flight_key after:          "
    f"{flight_keys_after:,}"
)

print(
    f"Weather airport-hour duplicates:    "
    f"{weather_duplicate_groups:,}"
)

print(
    f"Duplicate flight groups after join: "
    f"{duplicate_flight_groups_after:,}"
)

print()
print(
    f"Weather matched flights:            "
    f"{matched_rows:,}"
)

print(
    f"Weather unmatched flights:          "
    f"{unmatched_rows:,}"
)

print(
    f"ORD/ATL origin flights:             "
    f"{pilot_origin_rows:,}"
)

print(
    f"ORD/ATL matched flights:            "
    f"{pilot_matched_rows:,}"
)

print(
    f"ORD/ATL unmatched flights:          "
    f"{pilot_unmatched_rows:,}"
)

print(
    f"Outside-pilot origin flights:       "
    f"{outside_pilot_rows:,}"
)

print()
print(
    f"Wrong-airport matches:              "
    f"{wrong_airport_matches:,}"
)

print(
    f"Wrong-hour matches:                 "
    f"{wrong_hour_matches:,}"
)

print(
    f"CRSDepTime=2400 rows handled:       "
    f"{midnight_2400_rows:,}"
)

print()
print(
    "Join grain: "
    "1 accepted flight_key -> at most 1 origin weather row"
)

print(
    "Unmatched handling: "
    "LEFT JOIN preserves flight; weather columns remain NULL"
)

print(
    "Target table: "
    f"{TARGET_TABLE}"
)

print(
    "Metrics table: "
    f"{METRICS_TABLE}"
)

print("=" * 82)

print(
    "\nTASK 22 STATUS: PASS"
)


# ============================================================
# 16. HUMAN-READABLE EVIDENCE
# ============================================================

print(
    "\nMatched sample:"
)

display(
    target
    .filter(
        F.col(
            "origin_weather_match_status"
        )
        ==
        "MATCHED"
    )
    .select(
        "flight_key",
        "flight_date",
        "reporting_airline",
        "flight_number",
        "origin",
        "dest",
        "crs_dep_time_hhmm",
        "origin_sched_dep_local",
        "origin_sched_dep_hour_local",
        "origin_weather_hour_local",
        "origin_weather_hour_utc",
        "origin_temperature_2m_c",
        "origin_precipitation_mm",
        "origin_weather_match_status",
    )
    .orderBy(
        "flight_date",
        "origin",
        "origin_sched_dep_local",
    )
    .limit(25)
)


print(
    "\nPilot-airport unmatched sample:"
)

display(
    target
    .filter(
        F.col(
            "origin_weather_match_status"
        )
        ==
        "PILOT_AIRPORT_HOUR_UNMATCHED"
    )
    .select(
        "flight_key",
        "flight_date",
        "origin",
        "dest",
        "crs_dep_time_hhmm",
        "origin_sched_dep_local",
        "origin_sched_dep_hour_local",
        "origin_weather_match_status",
    )
    .orderBy(
        "flight_date",
        "origin_sched_dep_local",
    )
    .limit(25)
)


print(
    "\nEnrichment metrics:"
)

display(
    spark.table(
        METRICS_TABLE
    )
)

StatementMeta(, c53be6b9-12a1-4e21-81f8-df21613953fc, 3, Finished, Available, Finished, True)

TASK 22 CONFIGURATION
---------------------
Flight source : slv_flights_validated
Weather source: slv_weather_hourly
Target        : slv_flights_weather_enriched
Metrics       : slv_flights_weather_enrichment_metrics
Weather pilot : ['ORD', 'ATL']

Accepted flight rows:       597,919
Distinct flight_key values: 597,919
Weather rows:               1,440

SOURCE-GRAIN VALIDATION: PASS

Weather duplicate airport-hour groups: 0
Weather distinct keys: 1,440
Weather airports: {'ORD', 'ATL'}
WEATHER UNIQUENESS GATE: PASS

Derived timestamp NULL rows: 0
Invalid derived HH:MM rows: 0
Source CRSDepTime=2400 rows: 0
FLIGHT TIME DERIVATION: PASS

Flight rows before join: 597,919
Flight rows after join:  597,919
Distinct flight_key before: 597,919
Distinct flight_key after:  597,919
Duplicate flight_key groups after join: 0
POST-JOIN FLIGHT-GRAIN QA: PASS
WEATHER MATCH-CORRECTNESS QA: PASS

Weather matched flights:          59,813
Weather unmatched flights:        538,106
ORD/ATL origin flights:   

SynapseWidget(Synapse.DataFrame, fc50d990-52a0-482b-8f93-ce0e626bd690)


POST-WRITE TARGET QA: PASS

AIROPS 360 - TASK 22 WEATHER-TO-FLIGHT CARDINALITY-SAFE ENRICHMENT
Accepted flights before join:       597,919
Flights after weather join:         597,919
Distinct flight_key before:         597,919
Distinct flight_key after:          597,919
Weather airport-hour duplicates:    0
Duplicate flight groups after join: 0

Weather matched flights:            59,813
Weather unmatched flights:          538,106
ORD/ATL origin flights:             59,813
ORD/ATL matched flights:            59,813
ORD/ATL unmatched flights:          0
Outside-pilot origin flights:       538,106

Wrong-airport matches:              0
Wrong-hour matches:                 0
CRSDepTime=2400 rows handled:       0

Join grain: 1 accepted flight_key -> at most 1 origin weather row
Unmatched handling: LEFT JOIN preserves flight; weather columns remain NULL
Target table: slv_flights_weather_enriched
Metrics table: slv_flights_weather_enrichment_metrics

TASK 22 STATUS: PASS

Matched sample:


SynapseWidget(Synapse.DataFrame, 4dfd419f-1b0c-4fa7-afd8-cfc396fd7f4f)


Pilot-airport unmatched sample:


SynapseWidget(Synapse.DataFrame, 7aed4869-21f3-4e5f-b8a6-8a3456703037)


Enrichment metrics:


SynapseWidget(Synapse.DataFrame, 21ce0211-1c35-4f40-b419-dc1542a15b72)